In [68]:
%load_ext autoreload
%autoreload 2

In [32]:
import pandas as pd
import re

## Extracción del archivo

In [ ]:
ruta_archivo = "../data/unprocessed_files/SABI_Export_2.txt"
df = pd.read_csv(
    ruta_archivo, sep=";", index_col=0, encoding="utf-16",
    dtype={
        'Código primario CNAE 2009': str,
        'Código(s) segundario(s) CNAE 2009': str
    }
)
#, sep=";"  encoding="utf-16"
columns = list(df.columns)
columns_sql = ', '.join(columns)
placeholders = ", ".join(f":{c}" for c in columns)
print(columns)
print(columns_sql)
print(placeholders)


FileNotFoundError: [Errno 2] No such file or directory: '../data/unprocessed_files/SABI_Export_2.txt'

## Modificación del archivo

In [ ]:

# 1. Eliminamos columnas innecesarias para el estudio
# df = df.drop(columns=['Código consolidación', 'País'])

# 2. Cambiamos al formato fecha solo indicando el año
df['Ultimo año disponible'] = pd.to_datetime(df['Ultimo año disponible'], format='%d/%m/%Y', errors='coerce').dt.year
df['Fecha constitución'] = pd.to_datetime(df['Fecha constitución'], format='%d/%m/%Y', errors='coerce').dt.year

# 3. Renombre de todas las columnas
df = df.rename(columns={
    'Nombre': 'Company',
    'Código NIF': 'NIF_cod',
    'Calle' : 'Street',
    'Código postal' : 'ZIP_cod',
    'Localidad': 'City',
    'Provincia' : 'Province',
    'Código primario CNAE 2009' : 'CNAE_primary',
    'Código(s) segundario(s) CNAE 2009' : 'CNAE_secundary',
    'Número empleados Últ. año disp.' : 'N_emp',
    'Fecha constitución' : 'Date_Inc',
    'Ultimo año disponible' : 'Last_year',
    'Ingresos de explotación mil EUR Últ. año disp.' : 'Earnings_mil_last_year',
    'Coordenada - X' : "Longitude",
    'Coordenada - Y' : 'Latitude'
})

# 4. Limpiamos los valores nulos y adaptamos el texto a numerico
# df['Earnings_mil_last_year'] = df['Earnings_mil_last_year'].str.replace('.', '', regex=False)
df['Earnings_mil_last_year'] = pd.to_numeric(df['Earnings_mil_last_year'], errors='coerce')
df['N_emp'] = pd.to_numeric(df['N_emp'], errors='coerce')
not_geocoded = (df['Longitude'] == 0) & (df['Latitude'] == 0)
df.loc[not_geocoded, ['Longitude', 'Latitude']] = pd.NA

# 5. Limpieza de espacios y duplicados
columnas_texto = df.select_dtypes(include=['object','str']).columns
for col in columnas_texto:
    df[col] = df[col].astype(str).str.strip()

df = df.drop_duplicates(subset=['NIF_cod'])

In [ ]:
# 1. Cambia al formato fecha solo indicando el año
df['Last available year'] = pd.to_datetime(df['Last available year'], format='%d/%m/%Y', errors='coerce').dt.year
df['Date of Establishment'] = pd.to_datetime(df['Date of Establishment'], format='%d/%m/%Y', errors='coerce').dt.year

# 2. Renombre de todas las columnas
df = df.rename(columns={
    'NIF Code': 'NIF_Code',
    'CNAE 2009 Primary Code': 'CNAE_primary_Code',
    'CNAE 2009 Secondary Code(s)': 'CNAE_secundary_Code',
    'Number of employees Last avail. yr': 'N_emp',
    'Date of Establishment': 'Date_Est',
    'Last available year': 'Last_year',
    'Operating revenue / turnover th EUR Last avail. yr': 'Earnings_mil_last_year',
    'X coordinate' : 'Longitude',
    'Y coordinate' : 'Latitude'
})
df.columns = [c.replace(' ','_') for c in df.columns]

# 3. Limpieza de los valores nulos y adaptación del texto a numerico
df['Earnings_mil_last_year'] = pd.to_numeric(df['Earnings_mil_last_year'], errors='coerce')
df['N_emp'] = pd.to_numeric(df['N_emp'], errors='coerce')
not_geocoded = (df['Longitude'] == 0) & (df['Latitude'] == 0)
df.loc[not_geocoded, ['Longitude', 'Latitude']] = pd.NA

# 4. Limpieza de espacios y duplicados
text_columns = df.select_dtypes(include=['object', 'str']).columns
for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

df = df.drop_duplicates(subset=['NIF_Code'])

In [ ]:
print(df.columns.tolist())
#print(df['Código(s) segundario(s) CNAE 2009'].head(10))
faltantes = df['Latitude'].isna().sum()

print(f"Empresas sin coordenadas: {faltantes} de {len(df)} ({faltantes/len(df)*100:.1f}%)")

['Company', 'NIF_cod', 'Street', 'ZIP_cod', 'City', 'Province', 'CNAE_primary', 'CNAE_secundary', 'N_emp', 'Date_Inc', 'Last_year', 'Earnings_mil_last_year', 'Longitude', 'Latitude']
Empresas sin coordenadas: 16 de 14096 (0.1%)


## Exportación archivo limpio

In [ ]:
archivo_limpio = ruta_archivo.replace("ArchivosSinProcesar", "ArchivosProcesados").replace(".txt", "_limpio.csv")
df.to_csv(archivo_limpio, index=False, sep=";", encoding="utf-8-sig")

# ETL DATOS DE LA EMPRESA

In [ ]:
import pandas as pd

## Extracción de datos

In [ ]:
ruta_archivo = "../data/ArchivosSinProcesar/SABI_Export.xlsx"
df = pd.read_excel(ruta_archivo, index_col=0)
df.head(10)
print(df.columns[0])

Company Name


In [ ]:

metricas= [
    'Total activo',
    'Ingresos de explotación',
    'Importe neto Cifra de Ventas',
    'EBITDA',
    'EBIT',
    'Resultado del Ejercicio',
    'Ratio de liquidez',
    'Rentabilidad económica',
    'Rentabilidad financiera',
    'Endeudamiento'
    ]

df_final = pd.DataFrame()

for n in lista:
    columna = [col for col in df.columns if n in col]

    df_col = pd.melt(
    df,
    id_vars=['Código NIF'],          # La columna que se MANTIENE intacta por fila
    value_vars=columna,       # Las columnas que se van a convertir en filas (los años)
    var_name='Año',                   # Nombre de la nueva columna de años
    value_name=n     # Nombre de la nueva columna para los valores
    )

    df_col['Año'] = df_col['Año'].str.extract(r'(\d{4})').astype(int)

    df_final = pd.merge(
        df_final,
        df_col,
        on=['Código NIF', 'Año'],
        how='outer'   # 'outer' asegura que no se pierda ningún año/empresa aunque falte un dato en una de las dos
    )

    df_final = df_final.sort_values(
        by=['Código NIF', 'Año'],
        ascending=[True, False]
    ).reset_index(drop=True)

# 1. Identificamos las columnas de años (todas las que empiezan por "Total activo")
# Esto selecciona dinámicamente las columnas desde 2025 hasta 2001
columnas_activo = [col for col in df.columns if 'Total activo' in col]
columnas_ingresos = [col for col in df.columns if 'Ingresos de explotación' in col]
columnas_Ventas = [col for col in df.columns if 'Importe neto Cifra de Ventas' in col]
columnas_EBIT = [col for col in df.columns if 'EBIT' in col]
columnas_EBITDA = [col for col in df.columns if 'EBITDA' in col]
columnas_res_eje = [col for col in df.columns if 'Resultado del Ejercicio' in col]
columnas_marg_Ben = [col for col in df.columns if 'Margen de beneficio' in col]
columnas_liquidez = [col for col in df.columns if 'Ratio de liquidez' in col]
columnas_rent_econ = [col for col in df.columns if 'Rentabilidad económica' in col]
columnas_rent_fina = [col for col in df.columns if 'Rentabilidad financiera' in col]
columnas_endeudamiento = [col for col in df.columns if 'Endeudamiento' in col]

# 2. Aplicamos melt para despivotar la tabla
df_largo = pd.melt(
    df,
    id_vars=['Código NIF'],          # La columna que se MANTIENE intacta por fila
    value_vars=columnas_activo,       # Las columnas que se van a convertir en filas (los años)
    var_name='Año',                   # Nombre de la nueva columna de años
    value_name='Total_Activo_EUR'     # Nombre de la nueva columna para los valores
)

# 3. Limpiamos la columna 'Año' para quedarnos SOLO con el número del año (ej. '2025')
# Como SABI mete "Total activo\nmil EUR\n2025", extraemos los últimos 4 dígitos
df_largo['Año'] = df_largo['Año'].str.extract(r'(\d{4})').astype(int)

# 4. Ordenamos por Empresa (NIF) y luego por Año de forma descendente (de 2025 a 2001)
df_largo = df_largo.sort_values(by=['Código NIF', 'Año'], ascending=[True, False]).reset_index(drop=True)

print(df_largo.head(100))


   Código NIF   Año Total_Activo_EUR
0   A30106264  2025             n.d.
1   A30106264  2024       214067.165
2   A30106264  2023     221592.09064
3   A30106264  2022       211806.517
4   A30106264  2021       205784.355
..        ...   ...              ...
70  B30411318  2005        399.90745
71  B30411318  2004        293.88713
72  B30411318  2003        255.96951
73  B30411318  2002        201.38889
74  B30411318  2001        179.79995

[75 rows x 3 columns]


In [ ]:
metricas= [
    'Total activo',
    'Ingresos de explotación',
    'Importe neto Cifra de Ventas',
    'EBITDA',
    'EBIT',
    'Resultado del Ejercicio',
    'Ratio de liquidez',
    'Rentabilidad económica',
    'Rentabilidad financiera',
    'Endeudamiento'
    ]

patron_metricas = '|'.join(re.escape(m) for m in metricas)

# --- 3. Identificamos TODAS las columnas relevantes de una sola vez ---
columnas_metricas = [col for col in df.columns if re.search(patron_metricas, col)]

# --- 4. Un único melt para todas las métricas a la vez ---
df_largo = pd.melt(
    df,
    id_vars=['Código NIF'],
    value_vars=columnas_metricas,
    var_name='Columna_Original',
    value_name='Valor'
)

nombres_metricas = list(dict.fromkeys([
    c.split('\n')[0] for c in df.columns if len(c.split('\n')) >= 3 
]))
print(nombres_metricas)

['Total activo', 'Ingresos de explotación', 'Importe neto Cifra de Ventas', 'EBIT', 'EBITDA', 'Resultado del Ejercicio', 'Margen de beneficio (%)', 'Ratio de liquidez', 'Rentabilidad económica (%)', 'Rentabilidad financiera (%)', 'Endeudamiento (%)']


In [ ]:
'''metricas= [
    'Total activo',
    'Ingresos de explotación',
    'Importe neto Cifra de Ventas',
    'EBITDA',
    'EBIT',
    'Resultado del Ejercicio',
    'Ratio de liquidez',
    'Rentabilidad económica',
    'Rentabilidad financiera',
    'Endeudamiento'
    ]'''

# -- 1. Obtenemos todos los nombres de las variables financieras y de las variables financieras
#       que estan representadas en miles

metricas = list(dict.fromkeys([
    c.split('\n')[0] for c in df.columns if len(c.split('\n')) >= 3 
]))
print(nombres_metricas)

metricas = sorted(metricas, key=len, reverse=True)

metricas_en_miles = [
    m for m in metricas
    if any(('mil EUR' in col) and (m in col) for col in df.columns)
]


patron_metricas = '|'.join(re.escape(m) for m in metricas)

# --- 3. Identificamos TODAS las columnas relevantes de una sola vez ---
columnas_metricas = [col for col in df.columns if re.search(patron_metricas, col)]

# --- 4. Un único melt para todas las métricas a la vez ---
df_largo = pd.melt(
    df,
    id_vars=['Código NIF'],
    value_vars=columnas_metricas,
    var_name='Columna_Original',
    value_name='Valor'
)

# --- 5. Extraemos la métrica y el año en un solo paso con una regex de 2 grupos ---
extraido = df_largo['Columna_Original'].str.extract(
    rf'({patron_metricas}).*?(\d{{4}})',
    flags=re.DOTALL)
df_largo['Metrica'] = extraido[0]
df_largo['Año'] = extraido[1].astype(int)

# --- 6. Pivotamos: cada métrica pasa a ser su propia columna ---
df_final = df_largo.pivot_table(
    index=['Código NIF', 'Año'],
    columns='Metrica',
    values='Valor',
    aggfunc='first'   # 'first' porque no debería haber duplicados; evita errores si los hay
).reset_index()

df_final.columns.name = None  # Quita el nombre "Metrica" que pivot_table deja en el eje de columnas

# --- 7. Ordenamos una única vez, al final ---
df_final = df_final.sort_values(
    by=['Código NIF', 'Año'],
    ascending=[True, False]
).reset_index(drop=True)

columnas_valores = [c for c in df_final.columns if c not in ['Código NIF', 'Año']]
df_final[columnas_valores] = df_final[columnas_valores].apply(pd.to_numeric, errors='coerce')

for col in metricas_en_miles:
    df_final[col] = df_final[col] * 1000

df_final = df_final.dropna(subset=columnas_valores, how='all').reset_index(drop=True)

# --- Paso A: quitar cualquier sufijo entre paréntesis, ej. '(%)' ---
df_final.columns = [re.sub(r'\s*\([^)]*\)', '', c).strip() for c in df_final.columns]

# --- Paso B: sustituir espacios por guion bajo en todos los nombres de columna ---
df_final.columns = [c.replace(' ', '_') for c in df_final.columns]


['Total activo', 'Ingresos de explotación', 'Importe neto Cifra de Ventas', 'EBIT', 'EBITDA', 'Resultado del Ejercicio', 'Margen de beneficio (%)', 'Ratio de liquidez', 'Rentabilidad económica (%)', 'Rentabilidad financiera (%)', 'Endeudamiento (%)']


In [ ]:
print(df_final.head(10))
archivo_limpio = ruta_archivo.replace("ArchivosSinProcesar", "ArchivosProcesados").replace(".xlsx", "_limpio.csv")
df_final.to_csv(archivo_limpio, index=False, sep=";", encoding="utf-8-sig")

    NIF_Code  Year        EBIT      EBITDA  Economic_profitability  \
0  A30106264  2024  1842339.00  6227060.00                   0.863   
1  A30106264  2023  1492356.21  4772779.73                   0.283   
2  A30106264  2022 -1320547.00  1175210.00                  -0.054   
3  A30106264  2021  1125648.00  4032227.00                   0.532   
4  A30106264  2020   643478.00  4460040.00                   0.461   
5  A30106264  2019  4205939.00  8324911.00                   2.464   
6  A30106264  2018   558648.00  4597996.00                   0.236   
7  A30106264  2017  1493431.00  5204765.00                   0.771   
8  A30106264  2016  3624181.00  6821519.00                   4.956   
9  A30106264  2015  2456830.00  5899434.00                  -0.366   

   Financial_profitability  Indebtness  Liquidity_Ratio  \
0                    2.817      69.345            0.686   
1                    0.815      65.316            0.798   
2                   -0.137      60.455            0.

In [ ]:
# -- 2. Obtiene todos los nombres de las variables financieras y de las variables
#       que estan representadas en miles
metrics = list(dict.fromkeys([
    c.split('\n')[0] for c in df.columns if len(c.split('\n')) >= 3
]))

metrics = sorted(metrics, key=len, reverse=True)

metrics_in_thousands = [
    m for m in metrics
    if any(('th EUR' in col) and (m in col) for col in df.columns)
]

# Construye un patrón de búsqueda múltiple a partir de la lista de métricas.
metrics_pattern = '|'.join(re.escape(m) for m in metrics)

# -- 3. Identifica las columnas relevantes
metric_columns = [col for col in df.columns if re.search(metrics_pattern, col)]

# -- 4. Despivota los datos de métricas de la tabla (de columnas a filas): mantiene el NIF como
#        identificador fijo y transforma todas las columnas de métricas en una única columna
#        con su respectivo valor financiero.
df_melt = pd.melt(
    df,
    id_vars=['NIF Code'],
    value_vars=metric_columns,
    var_name='Original_Column',
    value_name='Value'
)

# -- 5. Extrae las metricas y el año de la columna unica y las separa en dos columnas
#       nuevas (Metric y Year)
extracted = df_melt['Original_Column'].str.extract(
    rf'({metrics_pattern}).*?(\d{{4}})',
    flags=re.DOTALL)
df_melt['Metric'] = extracted[0]
df_melt['Year'] = extracted[1].astype(int)

# -- 6. Pivota y cada métrica pasa a ser su propia columna, dejando las columnas de
#       NIF Code y Year
df_final = df_melt.pivot_table(
    index=['NIF Code', 'Year'],
    columns='Metric',
    values='Value',
    aggfunc='first'
).reset_index()

# Quita el nombre "Metric" que pivot_table deja en el eje de columnas
df_final.columns.name = None

# -- 7. Ordena las filas por NIF Code y Year
df_final = df_final.sort_values(
    by=['NIF Code', 'Year'],
    ascending=[True, False]
).reset_index(drop=True)

# Fuerza las columnas con metricas financieras a numero y las que no
# muestran valores como n.d. a NaN
value_columns = [c for c in df_final.columns if c not in ['NIF Code', 'Year']]
df_final[value_columns] = df_final[value_columns].apply(pd.to_numeric, errors='coerce')

# Multiplica por 1000 las métricas que venían en "th EUR" para dejarlas en el valor real
for col in metrics_in_thousands:
    df_final[col] = df_final[col] * 1000

# Elimina los años que no muestran ningun dato finacieros para la optimizacion del estudio
df_final = df_final.dropna(subset=value_columns, how='all').reset_index(drop=True)

# Quita cualquier sufijo entre paréntesis, ej. '(%)', sustituye barras y espacios por
# guion bajo en todos los nombres de las columnas
df_final.columns = [re.sub(r'\s*\([^)]*\)', '', c).strip() for c in df_final.columns]
df_final.columns = [c.replace('/', '_').replace(' ', '_') for c in df_final.columns]
df_final.columns = [re.sub(r'_+', '_', c).strip('_') for c in df_final.columns]

# DATOS GEOESPACIALES

In [41]:
# import osmnx as ox
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt


# Carga el archivo (leerá automáticamente el .dbf y .prj de la misma carpeta)
secciones = gpd.read_file('España_Seccionado2026/SECC_CE_20260101.shp')
df = pd.read_csv('poblacion_por_censo_Madrid.csv', index_col=False, sep=';', encoding='utf-8', thousands='.')
df.columns = df.columns.str.strip()

print(df.columns)

df['Total'] = df['Total'].astype(float)
secciones = secciones[secciones['CUSEC'] == '2807901001'].copy()


# secciones = secciones[secciones['NMUN'] == 'Madrid'].copy()
df = df[
    (df['Secciones'] == '2807901001 Madrid sección 01001') & 
    (df['Sexo'] == 'Total') & 
    (df['Nacionalidad'] == 'Total')
].copy()

area_km2 = secciones.area.iloc[0] / 1000000

df['Densidad_hab_km2'] = df['Total'] / area_km2

print(secciones.area.iloc[0])
print(area_km2)

print(df[['Secciones', 'Periodo', 'Total', 'Densidad_hab_km2']])

'''
print(secciones.columns)
print(secciones['geometry'].head())
print(secciones.area)
'''
# Dibuja el mapa rápidamente
secciones.plot()

plt.show()

DataSourceError: España_Seccionado2026/SECC_CE_20260101.shp: No such file or directory

## FUNCIONES

### Funcion renta per capital

In [1]:
import pandas as pd

df = pd.read_excel('../data/unprocessed_geodata_files/renta_poblacional_españa.xlsx', thousands='.', skiprows=7, nrows=55730, usecols='A:F', index_col=None)

df.rename(columns={
    ' ' : 'sections'
}, inplace=True)

df['sections'] = df['sections'].str.extract(r'^(\d{10})\b')
df = df.dropna(subset='sections').copy()
df = df[df['sections'].str.startswith('28079')].copy()

valores = [f for f in df.columns]

df_me = df.melt(
    id_vars=['sections'],
    value_vars=valores,
    var_name='year',
    value_name='per_capita_income'
)

df_me = df_me.sort_values(by=['sections','year'], ascending=[True,False]).reset_index(drop=True)

df_me['sections'] = df_me['sections'].astype(str).str.strip()
df_me['year'] = df_me['year'].astype(int)

df_me.set_index(['sections','year'],drop=False, inplace=True)




print(df_me.columns)
print(df_me.head(20))


print(df_me.loc['2807919044'])

df_me.to_csv("renta_poblacional_españa_prueba.csv", encoding='utf-8-sig', sep=';', index=False)


/Users/joticasmiras/Desktop/TFM/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Index(['sections', 'year', 'per_capita_income'], dtype='str')
                   sections  year per_capita_income
sections   year                                    
2807901001 2023  2807901001  2023             27408
           2022  2807901001  2022             26032
           2021  2807901001  2021             24134
           2020  2807901001  2020             25059
           2019  2807901001  2019             31119
2807901002 2023  2807901002  2023             20951
           2022  2807901002  2022             18878
           2021  2807901002  2021             19101
           2020  2807901002  2020             17175
           2019  2807901002  2019             17113
2807901003 2023  2807901003  2023             21921
           2022  2807901003  2022             19309
           2021  2807901003  2021             18421
           2020  2807901003  2020             18631
           2019  2807901003  2019             19085
2807901004 2023  2807901004  2023             32264
  

### Funcion Población

In [2]:
import pandas as pd

df_pob = pd.read_csv('../data/unprocessed_geodata_files/poblacion_por_censo_Madrid.csv', index_col=False, encoding='utf-8', sep=';')

df_pob['Secciones'] = df_pob['Secciones'].str.extract(r'^(\d{10})\b')
df_pob = df_pob.dropna(subset='Secciones').copy()
df_pob = df_pob[df_pob['Secciones'].str.startswith('28079')].copy()
df_pob = df_pob[(df_pob['Sexo'] == 'Total') & (df_pob['Nacionalidad'] == 'Total')].copy()

df_pob['Provincias'] = df_pob['Provincias'].str.replace(r'^\d+\s+', '', regex=True)
df_pob['Municipios'] = df_pob['Municipios'].str.replace(r'^\d+\s+', '', regex=True)

df_pob.drop(columns=["Sexo","Nacionalidad"], inplace=True)
df_pob.reset_index(drop=True, inplace=True)
df_pob.rename(columns={
    'Provincias' : 'provinces',
    'Municipios' : 'city',
    'Secciones' : 'sections',
    'Periodo' : 'year',
    'Total' : 'population'
}, inplace=True)

df_pob['sections'] = df_pob['sections'].astype(str).str.strip()
df_pob['year'] = df_pob['year'].astype(int)

df_pob.set_index(['sections', 'year'], drop=False, inplace=True)

print(df_pob.head(40))
print(df_pob.columns)
print(df_pob.tail())



                provinces    city    sections  year population
sections   year                                               
2807901001 2025    Madrid  Madrid  2807901001  2025      1.192
           2024    Madrid  Madrid  2807901001  2024      1.184
           2023    Madrid  Madrid  2807901001  2023      1.160
           2022    Madrid  Madrid  2807901001  2022      1.165
           2021    Madrid  Madrid  2807901001  2021      1.162
2807901002 2025    Madrid  Madrid  2807901002  2025        871
           2024    Madrid  Madrid  2807901002  2024        933
           2023    Madrid  Madrid  2807901002  2023        973
           2022    Madrid  Madrid  2807901002  2022        939
           2021    Madrid  Madrid  2807901002  2021        887
2807901003 2025    Madrid  Madrid  2807901003  2025      1.900
           2024    Madrid  Madrid  2807901003  2024      1.901
           2023    Madrid  Madrid  2807901003  2023      1.873
           2022    Madrid  Madrid  2807901003  2022    

/var/folders/_y/tq284_h51073h7gfv694vhfr0000gn/T/ipykernel_4802/1402359471.py:3: DtypeWarning: Columns (0: Total) have mixed types. Specify dtype option on import or set low_memory=False.
  df_pob = pd.read_csv('../data/unprocessed_geodata_files/poblacion_por_censo_Madrid.csv', index_col=False, encoding='utf-8', sep=';')


### Funcion densidad de poblacion

In [ ]:
# import osmnx as ox
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt


# Carga el archivo (leerá automáticamente el .dbf y .prj de la misma carpeta)
df_sections = gpd.read_file('../data/unprocessed_geodata_files/España_Seccionado2026/SECC_CE_20260101.shp')

df_sections.rename(columns={
    'CUSEC' : 'sections'
},inplace=True)

df_sections.set_index('sections', drop=False, inplace=True)

df_sections = df_sections[df_sections['sections'].str.startswith('28079')].copy()

df_sections['Area'] = df_sections.area

print(type(df_sections.area))

print(df_sections.head())

print(df_sections.loc['2807901001'])




<class 'pandas.Series'>
              sections  CUMUN CSEC CDIS CMUN CPRO CCA    CUDIS  CLAU2    NPRO  \
sections                                                                        
2807905058  2807905058  28079  058   05  079   28  13  2807905  28079  Madrid   
2807905059  2807905059  28079  059   05  079   28  13  2807905  28079  Madrid   
2807905060  2807905060  28079  060   05  079   28  13  2807905  28079  Madrid   
2807905061  2807905061  28079  061   05  079   28  13  2807905  28079  Madrid   
2807905062  2807905062  28079  062   05  079   28  13  2807905  28079  Madrid   

                            NCA CNUT0 CNUT1 CNUT2 CNUT3    NMUN  \
sections                                                          
2807905058  Comunidad de Madrid    ES     3     0     0  Madrid   
2807905059  Comunidad de Madrid    ES     3     0     0  Madrid   
2807905060  Comunidad de Madrid    ES     3     0     0  Madrid   
2807905061  Comunidad de Madrid    ES     3     0     0  Madrid   
280790

## DENSIDAD POBLACIONAL

In [42]:

# import osmnx as ox
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt


# Carga el archivo (leerá automáticamente el .dbf y .prj de la misma carpeta)
df_sections = gpd.read_file('../data/unprocessed_geodata_files/España_Seccionado2026/SECC_CE_20260101.shp')

df_sections.rename(columns={
    'CUSEC' : 'sections'
},inplace=True)

df_sections.set_index('sections', drop=False, inplace=True)

df_sections = df_sections[df_sections['sections'].str.startswith('28079')].copy()

df_sections['area_km2'] = df_sections.area / 1000000



df = pd.read_csv('../data/unprocessed_geodata_files/poblacion_por_censo_Madrid.csv', index_col=False, encoding='utf-8', sep=';', thousands='.')
df.columns = df.columns.str.strip()

print(df_sections.crs)

print(f'sections: {len(df_sections)}')

df['Total'] = df['Total'].astype(float)
df['Sexo'] = df['Sexo'].astype(str)
df['Nacionalidad'] = df['Nacionalidad'].astype(str)

df = df[
    (df['Secciones'].str.startswith('28079')) &
    (df['Sexo'] == 'Total') &
    (df['Nacionalidad'] == 'Total')
].copy()

df['Secciones'] = df['Secciones'].str.extract(r'^(\d{10})\b')

print(f'poblacion: {len(df)}')

df_final = pd.merge(
    df,
    df_sections['area_km2'],
    left_on='Secciones',
    right_on='sections',
    how='inner'
)

df_final['Densidad_hab_km2'] = df_final['Total'] / df_final['area_km2']

df_final.drop(columns=['Sexo','Nacionalidad'], inplace=True)

print(len(df))

print(df_final.head())

df_final.to_csv("prueba.csv", encoding='utf-8-sig', sep=';', index=False)
'''
print(secciones.columns)
print(secciones['geometry'].head())
print(secciones.area)
'''

EPSG:25830
sections: 2462
poblacion: 12480
12480
  Provincias    Municipios   Secciones  Periodo   Total  area_km2  \
0  28 Madrid  28079 Madrid  2807901001     2025  1192.0  0.657933   
1  28 Madrid  28079 Madrid  2807901001     2024  1184.0  0.657933   
2  28 Madrid  28079 Madrid  2807901001     2023  1160.0  0.657933   
3  28 Madrid  28079 Madrid  2807901001     2022  1165.0  0.657933   
4  28 Madrid  28079 Madrid  2807901001     2021  1162.0  0.657933   

   Densidad_hab_km2  
0       1811.734249  
1       1799.574958  
2       1763.097088  
3       1770.696644  
4       1766.136910  


"\nprint(secciones.columns)\nprint(secciones['geometry'].head())\nprint(secciones.area)\n"

## Numero de empresas por seccion

In [ ]:
import pandas as pd
import re

file_path = '../data/unprocessed_geodata_files/2024-censo-locales-madrid.csv'

df = pd.read_csv(file_path, sep=';', index_col=False, encoding='utf-8',
    dtype={
       'id_seccion_censal_local' : str 
    }
)

# Convierte a texto y rellena con ceros a la izquierda hasta garantizar 5 caracteres
df['id_seccion_censal_local'] = df['id_seccion_censal_local'].str.zfill(5)


df = df.groupby('id_seccion_censal_local').size().reset_index(name='num_company')

# Busca el año dentro de la ruta del archivo y lo asigna a las columnas
if match := re.search(r'\d{4}',file_path):
    df['Periodo'] = match.group()
df['Periodo'] = df['Periodo'].astype(int)

df['id_seccion_censal_local'] = '28079' + df['id_seccion_censal_local']

df.set_index(['id_seccion_censal_local', 'Periodo'], inplace=True)

print(len(df))

print(df.head(10))

df_merge = pd.merge(
    df_final,
    df,
    left_on=['Secciones','Periodo'],
    right_on=['id_seccion_censal_local','Periodo'],
    how= 'left'
)

print(df_merge.head())

df_merge.to_csv("prueba_merge.csv", encoding='utf-8-sig', sep=';', index=False)

2489
                                 num_company
id_seccion_censal_local Periodo             
2807901001              2024             206
2807901002              2024             250
2807901003              2024             626
2807901004              2024             139
2807901006              2024             279
2807901007              2024             117
2807901008              2024             230
2807901009              2024             204
2807901011              2024             155
2807901012              2024              95
  Provincias    Municipios   Secciones  Periodo   Total  area_km2  \
0  28 Madrid  28079 Madrid  2807901001     2025  1192.0  0.657933   
1  28 Madrid  28079 Madrid  2807901001     2024  1184.0  0.657933   
2  28 Madrid  28079 Madrid  2807901001     2023  1160.0  0.657933   
3  28 Madrid  28079 Madrid  2807901001     2022  1165.0  0.657933   
4  28 Madrid  28079 Madrid  2807901001     2021  1162.0  0.657933   

   Densidad_hab_km2  num_company  
0    

/var/folders/_y/tq284_h51073h7gfv694vhfr0000gn/T/ipykernel_4854/3983348508.py:6: DtypeWarning: Columns (0: id_planta_agrupado) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, sep=';', index_col=False, encoding='utf-8',


## Join DataFrames

In [ ]:



df_nuevo = df_me.set_index(['sections', 'year']).join(df_pob.set_index(['sections', 'year']), how='outer')

'''print(df_me)
print(df_pob)'''
'''print(df_nuevo)'''

df_nuevo.reset_index(inplace=True)

df_nuevo[['provinces','city']] = df_nuevo.groupby('sections')[['provinces','city']].bfill()

df_nuevo.to_csv("join_prueba.csv", encoding='utf-8-sig', sep=';', index=False)


In [25]:
import pandas as pd
import re
import os
import glob
import geopandas as gpd


def process_per_capita_income(file_path: str) -> pd.DataFrame:
    
    df = pd.read_excel(file_path, thousands='.', skiprows=7, nrows=55730, usecols='A:F', index_col=None)

    df.rename(columns={
        ' ' : 'sections'
    }, inplace=True)

    df['sections'] = df['sections'].str.extract(r'^(\d{10})\b')
    df = df.dropna(subset='sections').copy()
    df = df[df['sections'].str.startswith('28079')].copy()

    valores = [f for f in df.columns]
    df_melt = df.melt(
        id_vars=['sections'],
        value_vars=valores,
        var_name='year',
        value_name='per_capita_income'
    )
    df_melt = df_melt.sort_values(by=['sections','year'], ascending=[True,False]).reset_index(drop=True)

    df_melt['year'] = df_melt['year'].astype(int)
    df_melt['sections'] = df_melt['sections'].astype(str).str.strip()

    df_melt.set_index(['sections','year'], inplace=True)

    return df_melt


def process_population(file_path: str) -> pd.DataFrame:

    df_pob = pd.read_csv(file_path, thousands='.', index_col=False, encoding='utf-8', sep=';')

    df_pob['Secciones'] = df_pob['Secciones'].str.extract(r'^(\d{10})\b')
    df_pob = df_pob.dropna(subset='Secciones').copy()
    df_pob = df_pob[df_pob['Secciones'].str.startswith('28079')].copy()
    df_pob = df_pob[(df_pob['Sexo'] == 'Total') & (df_pob['Nacionalidad'] == 'Total')].copy()

    df_pob['Provincias'] = df_pob['Provincias'].str.replace(r'^\d+\s+', '', regex=True)
    df_pob['Municipios'] = df_pob['Municipios'].str.replace(r'^\d+\s+', '', regex=True)

    df_pob.drop(columns=["Sexo","Nacionalidad"], inplace=True)
    df_pob.reset_index(drop=True, inplace=True)
    df_pob.rename(columns={
        'Provincias' : 'provinces',
        'Municipios' : 'city',
        'Secciones' : 'sections',
        'Periodo' : 'year',
        'Total' : 'population'
    }, inplace=True)

    df_pob['sections'] = df_pob['sections'].astype(str).str.strip()
    df_pob['year'] = df_pob['year'].astype(int)
    df_pob['population'] = pd.to_numeric(df_pob['population'], errors='coerce')

    df_pob.set_index(['sections', 'year'], inplace=True)

    return df_pob


def process_num_company(input_folder: str) -> pd.DataFrame:

    files_path = [path for path in glob.glob(f'{input_folder}/*.csv')] 

    list_concat = []

    for file in files_path:
        df = pd.read_csv(file, sep=';', index_col=False, encoding='ISO-8859-1', on_bad_lines='skip',
        dtype={
            'id_seccion_censal_local' : str,
            'desc_situacion_local' : str,
            'desc_seccion_censal_local' : str,
            'id_distrito_local': str
        })

        df = df[df['desc_situacion_local'] == 'Abierto'].copy()

        # Convierte a texto y rellena con ceros a la izquierda hasta garantizar 5 caracteres
        df['desc_seccion_censal_local'] = df['desc_seccion_censal_local'].str.zfill(3)
        df['id_distrito_local'] = df['id_distrito_local'].str.zfill(2)

        df['sections'] = '28079' + df['id_distrito_local'] + df['desc_seccion_censal_local']

        df = df.groupby('sections').size().reset_index(name='num_company')

        # Busca el año dentro de la ruta del archivo y lo asigna a las columnas
        if match := re.search(r'\d{4}',file):
            df['year'] = match.group()
        df['year'] = df['year'].astype(int)
        df['sections'] = df['sections'].astype(str).str.strip()

        df.set_index(['sections', 'year'], inplace=True)

        list_concat.append(df)

    return pd.concat(list_concat)


def get_area_of_geodf(filepath : str) -> pd.DataFrame:

    file = glob.glob(f'{filepath}/*.shp')[0]

    df_sections = gpd.read_file(file)

    df_sections.rename(columns={
        'CUSEC' : 'sections'
    },inplace=True)

    df_sections = df_sections[df_sections['sections'].str.startswith('28079')].copy()

    # Proyectamos a metros
    if df_sections.crs != 'EPSG:25830':
        df_sections = df_sections.to_crs('EPSG:25830')

    df_sections['area_km2'] = df_sections.area / 1000000
    df_sections['sections'] = df_sections['sections'].astype(str).str.strip()


    return df_sections[['sections','area_km2']]


## Prueba de archivos que se obtienen

In [37]:
import glob
import os
import re

input_folder = '../data/unprocessed_geodata_files'

files_path = [
    path for path in glob.glob(f'{input_folder}/*')
    if not os.path.basename(path).startswith('~')
    ]

for path in files_path:
    print(path)
    if 'local' in path:
        df_num_companies = process_num_company(path)
    elif 'section' in path:
        df_area_section = get_area_of_geodf(path)
    elif 'population' in path:
        df_population = process_population(path)
    elif any(word in path for word in ['income','capita']):
        df_per_capita_income = process_per_capita_income(path)

df_final = df_population.join([df_num_companies, df_per_capita_income], how='outer')


df_final.head()




../data/unprocessed_geodata_files/local_census_madrid


/var/folders/_y/tq284_h51073h7gfv694vhfr0000gn/T/ipykernel_1342/2411514629.py:75: DtypeWarning: Columns (0: id_planta_agrupado) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, sep=';', index_col=False, encoding='ISO-8859-1', on_bad_lines='skip',
/var/folders/_y/tq284_h51073h7gfv694vhfr0000gn/T/ipykernel_1342/2411514629.py:75: DtypeWarning: Columns (0: id_planta_agrupado) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, sep=';', index_col=False, encoding='ISO-8859-1', on_bad_lines='skip',


../data/unprocessed_geodata_files/per_capita_income_spain.xlsx


/Users/joticasmiras/Desktop/TFM/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


../data/unprocessed_geodata_files/census_population_Madrid.csv
../data/unprocessed_geodata_files/spain_section_2025


provinces    city  population  num_company per_capita_income
sections   year                                                             
2807901001 2019       NaN     NaN         NaN          NaN             31119
           2020       NaN     NaN         NaN         84.0             25059
           2021    Madrid  Madrid      1162.0         80.0             24134
           2022    Madrid  Madrid      1165.0         82.0             26032
           2023    Madrid  Madrid      1160.0         83.0             27408

In [38]:
df_final[['provinces','city']] = df_final.groupby('sections')[['provinces','city']].bfill()

df_final.reset_index(level='year', inplace=True)

df_final_final = df_final.join(df_area_section.set_index(['sections']), how='outer')

df_final_final['densidad_hab_km2'] = (df_final_final['population'] / df_final_final['area_km2']).round(2)

df_final_final.drop(columns='area_km2', inplace=True)
df_final_final.reset_index(level='sections', inplace=True)


df_final_final.to_csv('pruebafinal.csv', sep=';', encoding='ISO-8859-1', index=False)

In [5]:
import pandas as pd
import re
import os
import glob

file = '../data/unprocessed_geodata_files/local_census_madrid/2020-censo-locales-madrid.csv'

df = pd.read_csv(file, sep=';', index_col=False, encoding='ISO-8859-1', on_bad_lines='skip',
    dtype={
        'id_seccion_censal_local' : str,
        'desc_situacion_local' : str,
        'desc_seccion_censal_local' : str,
        'id_distrito_local': str
    })
df = df[df['desc_situacion_local'] == 'Abierto'].copy()

df['desc_seccion_censal_local'] = df['desc_seccion_censal_local'].str.zfill(3)
df['id_distrito_local'] = df['id_distrito_local'].str.zfill(2)

df['sections'] = '28079' + df['id_distrito_local'] + df['desc_seccion_censal_local']

df = df.groupby('sections').size().reset_index(name='num_company')

# Busca el año dentro de la ruta del archivo y lo asigna a las columnas
if match := re.search(r'\d{4}',file):
    df['Periodo'] = match.group()
df['Periodo'] = df['Periodo'].astype(int)


df.set_index(['sections', 'Periodo'], drop=False, inplace=True)




df.head(20)

,,sections,num_company,Periodo
sections,Periodo,,,
2807901001,2020,2807901001,84,2020
2807901002,2020,2807901002,58,2020
2807901003,2020,2807901003,189,2020
2807901004,2020,2807901004,63,2020
2807901006,2020,2807901006,154,2020
2807901007,2020,2807901007,63,2020
2807901008,2020,2807901008,124,2020
2807901009,2020,2807901009,87,2020
2807901011,2020,2807901011,107,2020


# CAMBIO A DETECCION DE ARCHIVOS POR HASH

### HASH

In [15]:
import hashlib

file_path = 'join_prueba.csv'
chunk_size = 16384 # 16 KBytes
hash_f = hashlib.sha256()
with open(file_path, 'rb') as file:
    
    for c in iter(lambda: file.read(chunk_size), b''):
        hash_f.update(c)

    file_hash = hash_f.hexdigest()
print(file_hash)

38ac72fe0eafb3bf9a7c59588e617f65d5aa6f0b22b3928d0da21138a78dc7be


### Query SQL